**Notebook de procesamiento de imagenes hiperespectrales HSI usando submuestreo de selección de bandas y operaciones de visión por computadora**

El siguiente notebook tiene como finalidad cargar, submuestrear y aplicar técnicas de visón por computador a una imagen hiperespectral, donde la finalidad es transformar la imagen de hiperrespectral (HSI) a multiespectral (MSI). Logrando resaltar algunas características de interés de la imagen como zonas agricolas, edificios entre otros para lograr que la imagen pueda ser procesada por algún algoritmo de inteligencia artificial.

1. La imagen usada para procesar fue Indian Pines [2] donde esta cuenta con una dimesión en altura, ancho y profundidad es de (145,145,200).
2. El método de submuestreo de aplicado para la imagen fue el de selección de bandas con base a la longitud de onda del satelite LandSat 8, donde se seleccionaron las longitudes de onda usadas en [1]. La idea de submuestrear Indian Pines fue seleccionar solo nueve bandas para obtener una matriz del tamaño (145,145,9) es decir, transformar una imagen hiperespectral (HSI) a una multiespectral (MSI) para finalmente aplicarle las técnicas nombradas en el punto **3**.
3. Las técnicas de visión por computador aplicadas fueron las de entropía y operadores morfologicos usados en [3] y [4]. Donde la entropía mide el grado de incertidumbre o aleatoriedad en la información contenida en una imagen y los operadores morfológicos trabajan en la estructura o forma de los objetos dentro de una imagen.

Finalmente al aplicar las técnicas anteriores, se obtiene un realce de informacion en el aspecto espacial  de la imagen hiper o multiespectral para lograr clasificar materiales o regiones de interes de la imagen.

**Comenzamos**

Si vas a trabajar desde google colab se debe conceder los perimos a la nube para poder acceder a los datos. Se hace con la siguiente linea de codigo.

1. Preparación del dataset para submuestrear la imagen usando el método de selección de bandas.

By Camilo Acevedo-Correa


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Si estás trabajando de manera local omite el paso anterior

Importamos las liberías a usar

In [ ]:

import pandas as pd
import numpy as np
import scipy.io

import matplotlib.pyplot as plt

1. Cargar el dataset de longitud de onda que se desea usar para submuestrear las bandas de las imágenes. En este caso el archivo CSV con tiene todas las longitudes de onda de las 220 bandas de la HSI.

In [ ]:
df = pd.read_csv('wavelenght.csv',delimiter=';')

2.  Se revisa la información del dataset de la imagen correspondiente.

In [ ]:
df.info()

3. Se revisan los valores nulos del data set.

In [ ]:
df.isnull().sum()


In [ ]:
df.head()

4. Creanmos una copia del dataset por seguridad.

In [ ]:
copia = df.copy()
copia.shape

5. Eliminanos 19 bandas que no son necesarias en el dataset de longitud de onda de la imagen. 
 - Reiniciamos los indices de la tabla para conservar el orden en las filas.
 - Eliminamos las columnas index y data_channel y lo reemplazamos directamente en el dataframe.
 - Transformamos de dataframe a array usando el atributo values. este array es llamado w.
 - Tranponemos el array creado (w).

In [ ]:
df = df.drop(df.index[104:109].union(df.index[150:164]))
df = df.reset_index()
df.drop(['index','data_channel'], axis=1, inplace=True)

w = df.values
w = w.T
w.shape

6. Extrayendo la información de longitud de onda de referencia del satelite LandSat 8 disponible [1] fijamos las longitudes de onda de nuestro interes.
- Declaramos una lista para almacenar aquellas longitudes de onda disponibles en la rangos seleccionados.

In [ ]:

# Supongamos que tienes un vector w de longitud 200

# Define los rangos específicos que deseas buscar en w y divídelos en 9 sublistas
rangos = [(433, 453), (450, 515) ,(525, 600),(630, 680), (845, 885), (1560,1660),(2100, 2300), (500, 680), (1360,1390)]

sublistas_rangos = np.array_split(rangos, 9)  # Dividir la lista de rangos en 9 sublistas

# Encontrar los índices en w para cada sub-lista de rangos
indices_por_sublista = []
for sublist in sublistas_rangos:
    indices_sublista = []
    for rango in sublist:
        inicio, fin = rango
        indices_sublista.extend(np.where((w >= inicio) & (w <= fin))[1])
    indices_por_sublista.append(list(set(indices_sublista)))

# Ahora indices_por_sublista contiene 9 listas de índices en w correspondientes a cada sub-lista de rangos
print("Índices en w para cada sub-lista de rangos:")
print(indices_por_sublista)


7. Almacenamos las longitudes de onda en variables independientes.

In [ ]:
inx_band_1, inx_band_2, inx_band_3, inx_band_4, inx_band_5, inx_band_6, inx_band_7, inx_band_8, inx_band_9 = indices_por_sublista


8. Cargamos la imagen hiperespectral usando la librería scipy.

In [ ]:
image_path = 'Indian_pines_corrected.mat'

img = scipy.io.loadmat(image_path)
img = img['indian_pines_corrected']


9. Fusión de las longitudes de onda del punto **7** con la imagen hiperespectral.

In [ ]:
band_1 = img[:, :, inx_band_1]
band_2 = img[:, :, inx_band_2]
band_3 = img[:, :, inx_band_3]
band_4 = img[:, :, inx_band_4]
band_5 = img[:, :, inx_band_5]
band_6 = img[:, :, inx_band_6]
band_7 = img[:, :, inx_band_7]
band_8 = img[:, :, inx_band_8]
band_9 = img[:, :, inx_band_9]



10. Seleccionar cada banda según el índice tomado para un rango de longitudes de onda y promediar cada longitud de onda para obtener un solo valor.
- Concatenar cada banda obtenida con la imagen original
- Revisar si el submuestreo quedo adecuado.


In [ ]:
msi_1 = np.mean(band_1, axis=2,  keepdims=True)
msi_2 = np.mean(band_2, axis=2,  keepdims=True)
msi_3 = np.mean(band_3, axis=2,  keepdims=True)
msi_4 = np.mean(band_4, axis=2,  keepdims=True)
msi_5 = np.mean(band_5, axis=2,  keepdims=True)
msi_6 = np.mean(band_6, axis=2,  keepdims=True)
msi_7 = np.mean(band_7, axis=2,  keepdims=True)
msi_8 = np.mean(band_8, axis=2,  keepdims=True)
msi_9 = np.mean(band_9, axis=2,  keepdims=True)

msi_img = np.concatenate((msi_1, msi_2, msi_3, msi_4, msi_5, msi_6, msi_7, msi_8, msi_9), axis=2)
msi_img.shape


11. Importar la librería spectral para mostrar si la imagen quedó bien submuestreada.
- Si variamos cada uno de los parámetros de la función imshow del 1 al 9 podemos obtener diferentes longitudes de onda.
- para visualizar la imagen en RGB se seleccionan las bandas (3,2,1)

In [ ]:
from spectral import *

In [ ]:
view = imshow(msi_img,(3, 2, 1))

In [ ]:
view

msi 16 bits

In [ ]:
msi16=msi_img.astype(np.float16)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.io import imread, imshow
from skimage import data
from skimage.util import img_as_ubyte
from skimage.filters.rank import entropy
from skimage.morphology import disk
from skimage.color import rgb2hsv, rgb2gray, rgb2yuv

Seeking entropy

In [ ]:
multispectral_image =msi_img  


# Almacena cada banda en escala de grises en una variable
grayscale_bands = []

# Visualiza y almacena cada banda en escala de grises
plt.figure(figsize=(15, 10))

for i in range(multispectral_image.shape[2]):
    grayscale_band = multispectral_image[:, :, i]
    grayscale_bands.append(grayscale_band)

    plt.subplot(3, 3, i + 1)
    plt.imshow(grayscale_band)
    plt.title(f'Banda {i + 1}')

plt.show()

**Entropy method 1**

Técnicas de Entropía:


Descripción: La entropía se utiliza para medir el grado de incertidumbre o aleatoriedad en la información contenida en una imagen. En el preprocesamiento de imágenes, la entropía puede emplearse para:
- Evaluar la cantidad de información en diferentes regiones de una imagen.
- Detectar bordes o áreas de interés basadas en cambios en la entropía local.
- Mejorar el contraste en imágenes mediante técnicas de ajuste basadas en la entropía.

In [ ]:
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt
from skimage.morphology import disk

# Variables de entorno
Original Image of 9 channels is named **msi_img**.\
Entropic image of 9 channels is named **msi_img_entropic**\
Morphologic Image of 9 channels is named **msi_ip_morph**\
Entire Image of 27 channels is named **msi_ip_27**.

In [ ]:
msi_img.shape, type(msi_img), msi_img.dtype

Entropy method 2 medium.
https://towardsdatascience.com/image-processing-with-python-working-with-entropy-b05e9c84fc36

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.io import imread, imshow
from skimage import data
from skimage.util import img_as_ubyte
from skimage.filters.rank import entropy
from skimage.morphology import disk
from skimage.color import rgb2hsv, rgb2gray, rgb2yuv
from scipy.io import savemat


# Step 1
Normalization of data from multiespectral images. Transform matrix between 0 to 1.

In [ ]:

def normalize_image(image):
    # Obtener valores mínimo y máximo de la imagen
    min_val = np.min(image)
    max_val = np.max(image)

    # Normalizar la imagen al rango de 0 a 1
    normalized_image = (image - min_val) / (max_val - min_val)

    return normalized_image

In [ ]:
msi_img_norm = normalize_image(msi_img)

# Step 2
Process of entire array msi_img

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters.rank import entropy
from skimage.morphology import disk

def entropy_mask_viz(image_list):
    f_size = 24
    num_images = len(image_list)
    
    fig, axs = plt.subplots(num_images, 3, figsize=(17, 10*num_images))

    for i, image in enumerate(image_list):
        #normalize_image = normalize_image(image)
        entropy_image = entropy(normalize_image, disk(4))
        scaled_entropy = entropy_image / entropy_image.max()

        axs[i, 0].set_title(f'Greater Than Threshold - Image {i+1}', fontsize=f_size)
        threshold_a = scaled_entropy > 0.8
        image_a = image * threshold_a
        axs[i, 0].imshow(image_a)
        axs[i, 0].axis('off')




    fig.tight_layout()

    return [image_a]




In [ ]:
def entropy_mask_viz(image):
    # Calcular la entropía para cada banda espectral
    entropy_images = [entropy(image[:,:,i], disk(4)) for i in range(image.shape[2])]
    
    # Normalizar las imágenes de entropía
    scaled_entropy_images = [entropy_img / entropy_img.max() for entropy_img in entropy_images]
    
    f_size = 20
    fig, axs = plt.subplots(2,image.shape[2] + 1, figsize=(17, 10))
    e_msi = []
    for i in range(image.shape[2]):
        axs[0, i].set_title(f'Band {i+1}', fontsize=f_size)
        axs[0, i].imshow(image[:,:,i], cmap='gray')
        
        axs[0, i].axis('off')
        
        axs[1, i].set_title(f'E Band {i+1}', fontsize=f_size)
        axs[1, i].imshow(scaled_entropy_images[i], cmap='viridis')
        axs[1, i].axis('off')
    
    # Imágenes compuestas
    ax_comp_a = axs[0, -1]
    ax_comp_b = axs[1, -1]
    
    ax_comp_a.set_title('Greater Than Threshold', fontsize=f_size)
    threshold_a = scaled_entropy_images[0] > 0.7
    comp_image_a = np.sum(image * threshold_a[:, :, np.newaxis], axis=2)
    ax_comp_a.imshow(comp_image_a, cmap='viridis')
    ax_comp_a.axis('off')

    ax_comp_b.set_title('Less Than Threshold', fontsize=f_size)
    threshold_b = scaled_entropy_images[0] < 0.8
    comp_image_b = np.sum(image * threshold_b[:, :, np.newaxis], axis=2)
    ax_comp_b.imshow(comp_image_b, cmap='viridis')
    ax_comp_b.axis('off')
    
    fig.tight_layout()
    
    return e_msi

# Uso de la función
entropy_mask_viz(msi_img_norm)
plt.show()

# Nota: Entre mas pequeño es el radio del disco mas finos son los detalles que capta la entropía; si es mas grande suavizara esos detalles finos generando una entropía mas grande.

In [ ]:
def entropy_mask_viz(image):
    #Normalizar imagen
    image=normalize_image(image)
    # Calcular la entropía para cada banda espectral
    entropy_images = [entropy(image[:,:,i], disk(4)) for i in range(image.shape[2])]
    
    # Normalizar las imágenes de entropía
    scaled_entropy_images = [entropy_img / entropy_img.max() for entropy_img in entropy_images]
    
    # Crear una matriz para almacenar las imágenes de entropía normalizadas
    e_msi = np.zeros_like(image, dtype=float)
    
    for i in range(image.shape[2]):
        # Almacenar las imágenes de entropía normalizadas en la matriz e_msi
        e_msi[:,:,i] = scaled_entropy_images[i]
    
    # Imágenes compuestas
    threshold_a = scaled_entropy_images[0] > 0.7
    comp_image_a = np.sum(image * threshold_a[:, :, np.newaxis], axis=2)

    threshold_b = scaled_entropy_images[0] < 0.8
    comp_image_b = np.sum(image * threshold_b[:, :, np.newaxis], axis=2)
        
    return e_msi


msi_img_entropic = entropy_mask_viz(msi_img_norm)

In [ ]:
msi_img_entropic.shape, type(msi_img_entropic), msi_img_entropic.dtype

In [ ]:
msi_ip = np.concatenate((msi_img,msi_img_entropic),axis=2)
msi_ip.shape, type(msi_ip)

In [ ]:
file_name = 'IP_18'

savemat(f'{file_name}.mat', {'msi_ip': msi_ip})

In [ ]:
def show_matrix(image):
    #image_gray = rgb2gray(image)
    s =image.shape[2]
    f_size = 20
    #radi = list(range(1,s+1))
    fig, ax = plt.subplots(3,int(s/3),figsize=(15,15))
    for n, ax in enumerate(ax.flatten()):
        ax.set_title(f'Band {[n]}', fontsize = f_size)
        ax.imshow(image[:,:,n])
        ax.set_axis_off()
    fig.tight_layout()

look = show_matrix(msi_ip)

In [ ]:
image_path_18 = 'IP_18.mat'

img_18 = scipy.io.loadmat(image_path_18)
img_18 = img_18['msi_ip']


In [ ]:
img_18.shape

In [ ]:
show_matrix(img_18)

 

# Operadores Morfológicos:

Clasificación: Métodos de Procesamiento Morfológico.
Descripción: Los operadores morfológicos trabajan en la estructura o forma de los objetos dentro de una imagen. Estos operadores incluyen:
- Dilatación: Aumenta el tamaño de las regiones blancas (o regiones de interés) en una imagen binaria.
- Erosión: Reduce el tamaño de las regiones blancas en una imagen binaria.
- Apertura: Suaviza los contornos de una imagen, elimina pequeños objetos y separa objetos conectados.
- Clausura: Cierra pequeños huecos y conecta pequeños elementos dispersos.
- Aplicaciones: Los operadores morfológicos son utilizados para la segmentación de imágenes, eliminación de ruido, y mejora de la estructura de los objetos en una imagen.

- Finding morphologic
https://github.com/j-manansala/morphological/blob/main/Morphological%20Operations.ipynb

In [ ]:
from skimage.morphology import dilation, erosion, opening, closing, area_opening, area_closing


1. Definamos un elemento estructurante. Observe que utilizaremos un elemento de 5x5 debido a la forma más grande de la imagen real. Una regla general para definir el elemento estructurador es observar los objetos que desea eliminar y los que desea conservar. La forma del elemento estructurador debe ser mayor que los píxeles de los objetos que desea eliminar y menor que los píxeles de los objetos que desea conservar.
2. A partir de este elemento estructurante, podemos aplicar sucesivas operaciones de erosión para eliminar ruido de la imagen.
3. Tras la erosión sucesiva, la mayoría de los objetos y artefactos innecesarios se han erosionado. Sin embargo, observe que en algunas zonas sigue habiendo un ruido diminuto. Podemos eliminarlo aplicando la operación de apertura.
4. Ahora, como hemos aplicado una erosión sucesiva, el tamaño y la forma de los objetos son menores que los del original. Para restaurar aproximadamente su tamaño, tenemos que aplicar dilataciones sucesivas con el mismo número de veces que hemos aplicado la erosión. Además, debemos utilizar el mismo elemento estructurador para garantizar que la restauración de la forma de los rasgos sea lo más parecida posible al original.





In [ ]:
def procesar_msi(msi_ip):
    element = np.ones([5, 5])
    def multi_dil(im, num, element=element):

        for i in range(num):
            im = dilation(im, element)
        return im

    def multi_ero(im, num, element=element):
        for i in range(num):
            im = erosion(im, element)
        return im

    # Convertir el vector msi_ip a una matriz tridimensional (imagen)
    #msi_img = np.array([msi_ip], dtype=np.uint8)

    # Procesar cada canal por separado
    processed_channels = []
    for channel in range(msi_img.shape[2]):
        channel_data = msi_img[:,:, channel]
        multi_eroded = multi_ero(channel_data, 2, element)
        opened = opening(multi_eroded, element)
        multi_diluted = multi_dil(opened, 2, element)
        area_morphed = area_opening(area_closing(multi_diluted, 1000), 1000)
        processed_channels.append(area_morphed)

    # Combinar los canales procesados para obtener la imagen final
    #final_result = np.stack(processed_channels, axis=2)[0]

    return processed_channels

# Ejemplo de uso:

resultado_final = procesar_msi(msi_ip)

# Visualizar la imagen resultant

In [ ]:
msi_ip_morph = np.array(resultado_final)
msi_ip_morph = np.transpose(msi_ip_morph, (1,2,0))
type(resultado_final), msi_ip_morph.shape,msi_img.shape

In [ ]:
show_matrix(msi_ip_morph)

# Final Step
Concat msi_ip + msi_ip_entropic + msi_ip_morph

In [ ]:
msi_ip_27 = np.concatenate((msi_ip,msi_ip_morph),axis=2)
msi_ip_27.shape, type(msi_ip_27)

In [ ]:
file_name = 'IP_18'

savemat(f'{file_name}.mat', {'IP_18': msi_ip_27})

In [ ]:
image_path_27 = 'IP_18.mat'

img_27 = scipy.io.loadmat(image_path_18)
img_27 = img_27['IP_18']

img_27.shape

In [ ]:
show_matrix(img_27)

**Bibliografía**
- [1] https://landsat.gsfc.nasa.gov/satellites/landsat-8/
- [2] https://www.ehu.eus/ccwintco/index.php/Hyperspectral_Remote_Sensing_Scenes#Indian_Pines
- [3] https://towardsdatascience.com/image-processing-with-python-working-with-entropy-b05e9c84fc36
- [4] https://github.com/j-manansala/morphological/blob/main/Morphological%20Operations.ipynb
